# Probability Calibration

This notebook turns the notes into runnable code, using **scikit-learn's built-in
implementations directly — no user-defined functions anywhere**.

Covers:
1. `predict()` vs `predict_proba()`
2. Accuracy vs. calibration (same prediction, different confidence)
3. Calibration curve / reliability diagram
4. Brier score
5. Calibrating a model: Platt scaling (sigmoid) vs. Isotonic regression (`CalibratedClassifierCV`)
6. Uniform vs. quantile binning, and the effect of number of bins
7. Metrics comparison: accuracy, precision, recall, ROC-AUC, log loss, Brier score


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve, CalibrationDisplay, CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, roc_auc_score,
    log_loss, brier_score_loss,
)

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 6)


## 2. Synthetic Dataset

A binary classification problem (e.g. "does a student drop out?"), split into
train / calibration / test sets. `GaussianNB` is used as the base model — it's a
classic example of a model that tends to be **overconfident** (poorly calibrated)
even though it can be reasonably accurate.


In [ ]:
X, y = make_classification(
    n_samples=4000, n_features=10, n_informative=5, n_redundant=2,
    weights=[0.7, 0.3], class_sep=0.8, random_state=42,
)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
X_cal, X_test, y_cal, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("Train:", X_train.shape, " Calibration:", X_cal.shape, " Test:", X_test.shape)


## 3. Train the Base Model

In [ ]:
base_model = GaussianNB()
base_model.fit(X_train, y_train)


## 4. `predict()` vs. `predict_proba()`

`predict()` returns the class label; `predict_proba()` returns probabilities for
every class. `model.predict_proba(X_test)[:, 1]` extracts only the positive-class probability.


In [ ]:
predicted_labels = base_model.predict(X_test)
predicted_probabilities = base_model.predict_proba(X_test)
positive_class_probability = base_model.predict_proba(X_test)[:, 1]

pd.DataFrame({
    "predict() -> label": predicted_labels[:5],
    "predict_proba() -> P(class=0)": predicted_probabilities[:5, 0],
    "predict_proba() -> P(class=1)": predicted_probabilities[:5, 1],
    "positive_class_probability": positive_class_probability[:5],
})


## 5. Accuracy vs. Probability Calibration

Two models can make the **same classification** at a 0.5 threshold while having very
different confidence. Here, `LogisticRegression` and `GaussianNB` are compared on the
same test point range — accuracy only looks at the thresholded label, calibration
looks at the actual probability values.


In [ ]:
logreg_model = LogisticRegression(max_iter=1000)
logreg_model.fit(X_train, y_train)

logreg_proba = logreg_model.predict_proba(X_test)[:, 1]
gnb_proba = base_model.predict_proba(X_test)[:, 1]

logreg_labels = (logreg_proba >= 0.5).astype(int)
gnb_labels = (gnb_proba >= 0.5).astype(int)

agreement_rate = (logreg_labels == gnb_labels).mean()

comparison_snapshot = pd.DataFrame({
    "LogisticRegression P(y=1)": logreg_proba[:8],
    "GaussianNB P(y=1)": gnb_proba[:8],
    "LogReg predicted class": logreg_labels[:8],
    "GNB predicted class": gnb_labels[:8],
})
print(f"Same predicted class in {agreement_rate:.1%} of test cases, despite different confidence levels.")
comparison_snapshot


## 6. Calibration Curve / Reliability Diagram

`sklearn.calibration.calibration_curve` bins predictions and, for each bin, reports
the mean predicted probability against the observed fraction of positives.
`CalibrationDisplay` plots this directly against the perfect diagonal.


In [ ]:
fraction_of_positives_gnb, mean_predicted_value_gnb = calibration_curve(
    y_test, gnb_proba, n_bins=10, strategy="uniform"
)
fraction_of_positives_logreg, mean_predicted_value_logreg = calibration_curve(
    y_test, logreg_proba, n_bins=10, strategy="uniform"
)

fig, ax = plt.subplots(figsize=(7, 6))
CalibrationDisplay.from_predictions(y_test, gnb_proba, n_bins=10, name="GaussianNB (base model)", ax=ax)
CalibrationDisplay.from_predictions(y_test, logreg_proba, n_bins=10, name="LogisticRegression", ax=ax)
ax.set_title("Reliability Diagram: Base Models")
plt.tight_layout()
plt.show()


**Reading the diagram:**
- Curve **near the diagonal** → well-calibrated.
- Curve **below the diagonal** (predicted > actual) → overconfidence.
- Curve **above the diagonal** (predicted < actual) → underprediction.


## 7. Brier Score

$$\text{Brier Score} = \frac{1}{N}\sum_{i=1}^{N}(P_i - y_i)^2$$

Lower is better (0 = perfect). Computed directly with `brier_score_loss`.


In [ ]:
brier_gnb = brier_score_loss(y_test, gnb_proba)
brier_logreg = brier_score_loss(y_test, logreg_proba)

pd.Series({"GaussianNB": brier_gnb, "LogisticRegression": brier_logreg}, name="Brier Score")


## 8. Calibrating the Model

`CalibratedClassifierCV` handles the whole calibration workflow directly — including
generating out-of-fold predictions via cross-validation so calibration isn't fit on
the same data the base model was trained on. Both calibration methods from the notes
are used: **Platt scaling (`sigmoid`)** and **Isotonic regression (`isotonic`)**.


In [ ]:
sigmoid_calibrated_model = CalibratedClassifierCV(GaussianNB(), method="sigmoid", cv=5)
sigmoid_calibrated_model.fit(X_train, y_train)

isotonic_calibrated_model = CalibratedClassifierCV(GaussianNB(), method="isotonic", cv=5)
isotonic_calibrated_model.fit(X_train, y_train)

sigmoid_calibrated_proba = sigmoid_calibrated_model.predict_proba(X_test)[:, 1]
isotonic_calibrated_proba = isotonic_calibrated_model.predict_proba(X_test)[:, 1]


### Reliability Diagram: Before vs. After Calibration

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
CalibrationDisplay.from_predictions(y_test, gnb_proba, n_bins=10, name="Uncalibrated GaussianNB", ax=ax)
CalibrationDisplay.from_predictions(y_test, sigmoid_calibrated_proba, n_bins=10, name="Sigmoid (Platt) Calibrated", ax=ax)
CalibrationDisplay.from_predictions(y_test, isotonic_calibrated_proba, n_bins=10, name="Isotonic Calibrated", ax=ax)
ax.set_title("Reliability Diagram: Before vs. After Calibration")
plt.tight_layout()
plt.show()


In [ ]:
brier_sigmoid = brier_score_loss(y_test, sigmoid_calibrated_proba)
brier_isotonic = brier_score_loss(y_test, isotonic_calibrated_proba)

pd.Series({
    "Uncalibrated GaussianNB": brier_gnb,
    "Sigmoid (Platt) Calibrated": brier_sigmoid,
    "Isotonic Calibrated": brier_isotonic,
}, name="Brier Score").sort_values()


## 9. Uniform vs. Quantile Binning

- **Uniform**: equal-width bins (0.0-0.1, 0.1-0.2, ...) — consistent probability ranges, but high-probability bins can end up with very few observations.
- **Quantile**: roughly equal numbers of observations per bin — more stable estimates when predictions cluster in certain ranges.


In [ ]:
fraction_uniform, mean_pred_uniform = calibration_curve(y_test, gnb_proba, n_bins=10, strategy="uniform")
fraction_quantile, mean_pred_quantile = calibration_curve(y_test, gnb_proba, n_bins=10, strategy="quantile")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
CalibrationDisplay.from_predictions(y_test, gnb_proba, n_bins=10, strategy="uniform", name="Uniform bins", ax=axes[0])
axes[0].set_title("Uniform Binning")
CalibrationDisplay.from_predictions(y_test, gnb_proba, n_bins=10, strategy="quantile", name="Quantile bins", ax=axes[1])
axes[1].set_title("Quantile Binning")
plt.tight_layout()
plt.show()


## 10. Effect of Number of Bins

- **Fewer bins**: more observations per bin, more stable, less detail.
- **More bins**: more detail, but fewer samples per bin and noisier estimates.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
CalibrationDisplay.from_predictions(y_test, gnb_proba, n_bins=5, name="n_bins=5", ax=axes[0])
axes[0].set_title("n_bins = 5 (stable, less detail)")
CalibrationDisplay.from_predictions(y_test, gnb_proba, n_bins=10, name="n_bins=10", ax=axes[1])
axes[1].set_title("n_bins = 10 (common default)")
CalibrationDisplay.from_predictions(y_test, gnb_proba, n_bins=20, name="n_bins=20", ax=axes[2])
axes[2].set_title("n_bins = 20 (more detail, more noise)")
plt.tight_layout()
plt.show()


## 11. Metrics Comparison

Each metric answers a different question. Discrimination metrics (accuracy, ROC-AUC)
can look great even when calibration is poor.


In [ ]:
metrics_df = pd.DataFrame({
    "Uncalibrated GaussianNB": {
        "Accuracy": accuracy_score(y_test, (gnb_proba >= 0.5).astype(int)),
        "Precision": precision_score(y_test, (gnb_proba >= 0.5).astype(int)),
        "Recall": recall_score(y_test, (gnb_proba >= 0.5).astype(int)),
        "ROC-AUC": roc_auc_score(y_test, gnb_proba),
        "Log Loss": log_loss(y_test, gnb_proba),
        "Brier Score": brier_score_loss(y_test, gnb_proba),
    },
    "Sigmoid Calibrated": {
        "Accuracy": accuracy_score(y_test, (sigmoid_calibrated_proba >= 0.5).astype(int)),
        "Precision": precision_score(y_test, (sigmoid_calibrated_proba >= 0.5).astype(int)),
        "Recall": recall_score(y_test, (sigmoid_calibrated_proba >= 0.5).astype(int)),
        "ROC-AUC": roc_auc_score(y_test, sigmoid_calibrated_proba),
        "Log Loss": log_loss(y_test, sigmoid_calibrated_proba),
        "Brier Score": brier_score_loss(y_test, sigmoid_calibrated_proba),
    },
    "Isotonic Calibrated": {
        "Accuracy": accuracy_score(y_test, (isotonic_calibrated_proba >= 0.5).astype(int)),
        "Precision": precision_score(y_test, (isotonic_calibrated_proba >= 0.5).astype(int)),
        "Recall": recall_score(y_test, (isotonic_calibrated_proba >= 0.5).astype(int)),
        "ROC-AUC": roc_auc_score(y_test, isotonic_calibrated_proba),
        "Log Loss": log_loss(y_test, isotonic_calibrated_proba),
        "Brier Score": brier_score_loss(y_test, isotonic_calibrated_proba),
    },
}).T

metrics_df


Notice that **accuracy and ROC-AUC barely change** after calibration — calibration
doesn't improve discrimination, only probability reliability (Brier score / log loss
and the reliability diagram shape).


## 12. Common Calibration Mistakes

- **Calibrating on the final test set** — fitting the calibrator using test labels, then
  reporting performance on that same test set, leaks evaluation information. This
  notebook keeps calibration (`CalibratedClassifierCV`'s internal CV folds on
  `X_train`/`y_train`) and evaluation (`X_test`/`y_test`) fully separate.
- **Assuming isotonic is always better** — it's more flexible than sigmoid/Platt
  scaling, so it can overfit when the calibration set is small.


## 13. Key Takeaways

- `predict()` gives the class label; `predict_proba()` gives the full probability distribution — use `[:, 1]` for the positive class in binary problems.
- A model can have high accuracy but poor calibration — accuracy only sees the thresholded label, not the confidence behind it.
- A reliability diagram plots mean predicted probability vs. observed frequency per bin; the diagonal is perfect calibration.
- Curve below the diagonal → overconfidence; curve above the diagonal → underprediction.
- Brier score is a single numeric summary of probability quality, but it blends calibration *and* discrimination — pair it with the reliability diagram rather than reading it alone.
- `CalibratedClassifierCV` implements both **Platt scaling (sigmoid)** and **Isotonic regression** directly, using cross-validated out-of-fold predictions.
- Bin choice matters: fewer bins = more stable but less detail; more bins = more detail but noisier. Quantile binning helps when predictions cluster in a narrow range.
- Calibration improves probability reliability — it does **not** improve accuracy.
